# Génération

## Fil rouge

Le modèle est entraîné. Dernière étape : l'utiliser pour générer du texte. Vous allez implémenter une fonction de génération autoregressive.

## Contexte

Le modèle prédit le token suivant. Pour générer du texte : on part d'un prompt, on prédit le prochain token, on l'ajoute au contexte, et on recommence jusqu'à atteindre la longueur souhaitée. C'est la génération autoregressive.

## Ce qu'on attend de vous

- Comprendre le sampling et l'effet de la température
- Produire du texte généré par le modèle
- Expliquer l'effet de la température (diversité vs cohérence)

## Comment faire

1. Chargez le modèle sauvegardé
2. Implémentez une fonction generate(prompt, max_tokens, temperature)
3. La température contrôle le sampling : basse = plus déterministe, haute = plus varié
4. Testez avec différents prompts et températures

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Chargez le modèle (réutilisez la définition du notebook 03)
checkpoint = torch.load('model_simple.pt', map_location='cpu')
char_to_id = checkpoint['char_to_id']
id_to_char = {i: c for c, i in char_to_id.items()}
vocab_size = len(char_to_id)

# Recréez le modèle (même définition que notebook 03) et chargez les poids
class SimpleLLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.fc = nn.Linear(embed_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)
    def forward(self, x):
        h = self.embed(x)
        h = torch.relu(self.fc(h))
        return self.out(h)

model = SimpleLLM(vocab_size)
model.load_state_dict(checkpoint['model'])

# A COMPLETER : Fonction de génération
def generate(model, prompt, max_tokens=100, temperature=1.0):
    """
    Génère du texte à partir d'un prompt.
    - prompt : chaîne de caractères de départ
    - max_tokens : nombre de tokens à générer
    - temperature : contrôle la diversité (0 = déterministe, >1 = plus aléatoire)
    """
    model.eval()
    tokens = [char_to_id.get(c, 0) for c in prompt]
    
    with torch.no_grad():
        for _ in range(max_tokens):
            x = torch.tensor([tokens[-32:]], dtype=torch.long)  # Derniers 32 tokens
            logits = model(x)[0, -1, :]  # Logits du dernier token
            probs = F.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            tokens.append(next_token)
    
    return ''.join([id_to_char.get(t, '?') for t in tokens])

# A COMPLETER : Testez generate() avec différents prompts et températures
# Exemple : 
print(generate(model, "From ", max_tokens=100, temperature=0.8))

From sst t pof uthintit thin rind tothy ls thitorir line ine ththerthe d an t,
OIeme.
An ain we thy sund 


## À la fin de ce notebook, vous devez être capable de :

- Générer du texte à partir d'un prompt
- Expliquer l'effet de la température sur la diversité du texte généré